# Modern JavaScript: The Efficiency Upgrade

A practical deep-dive into modern JavaScript features that make everyday code more expressive and efficient. Covers strings, control flow, error handling, iteration patterns, data collections, and functional programming essentials — all with working Node.js demos.

## Table of Contents

- [Module 01: Strings](#module-01-strings)
- [Module 02: Control Flow](#module-02-control-flow)
- [Module 03: Error Handling](#module-03-error-handling)
- [Module 04: Iteration Patterns](#module-04-iteration-patterns)
- [Module 05: Data Collections](#module-05-data-collections)
- [Module 06: Functional Programming Essentials](#module-06-functional-programming-essentials)

## Module 01: Strings

Modern JavaScript provides template literals, tagged templates, and a rich set of string prototype methods. This module also covers Unicode handling, which is essential for working correctly with emoji and international characters.

### Template Literals

Template literals (backtick strings) support embedded expressions via `${}` and span multiple lines naturally. They replace string concatenation for anything beyond trivial joins.

```javascript
const firstName = 'Kim';
const cartItem = {
  event: 'The Firewall Squad',
  quantity: 1,
  price: 42
};

// Single expression
const message = `Hello ${firstName}!`;

// Multi-line + expressions + ternary
const fullMessage = `Hello ${firstName}!
Your cart contains ${cartItem.quantity} ticket${cartItem.quantity > 1 ? 's' : ''} for ${cartItem.event}
for a total of $${(cartItem.quantity * cartItem.price).toFixed(2)}.`;
```

Any valid JavaScript expression can appear inside `${}`, including ternary operators and method calls.

### Tagged Templates

A tagged template calls a function with the static string parts and interpolated values as separate arguments. This enables custom formatting, escaping, or HTML generation without string concatenation.

```javascript
function formatPlaceholders(strings, ...values) {
    return strings.reduce((result, part, pos) => {
        const value = (pos < values.length) ? `<em>${values[pos]}</em>` : '';
        return result + part + value;
    }, '');
}

const quantity = 2;
const event = 'The Firewall Squad';

// The tag receives:
//   strings = ['', ' ticket(s) for ', '.']
//   values  = [quantity, event]
const message = formatPlaceholders`${quantity} ticket(s) for ${event}.`;
// → '<em>2</em> ticket(s) for <em>The Firewall Squad</em>.'
```

The tag function receives `strings` (a frozen array of literal parts) and rest parameter `values` (the evaluated expressions). `strings.length` is always `values.length + 1`.

### Modern String Methods

Key `String.prototype` methods for everyday use:

```javascript
const s = ' --> Globoticket <-- ';

s.trim()                        // '--> Globoticket <--'
s.slice(5, 16)                  // 'Globoticket'
s.includes('bot')               // true  (case-sensitive)
s.substring(5).startsWith('Glob') // true
```

Additional useful methods:

| Method | Purpose |
|--------|---------|
| `match()` / `matchAll()` | Regex search; `matchAll` returns an iterator of all matches |
| `replace()` / `replaceAll()` | String/regex substitution |
| `padStart()` / `padEnd()` | Pad to a target length (useful for formatting) |
| `repeat(n)` | Repeat the string `n` times |
| `split(sep)` | Split into an array |

### Unicode in JavaScript

JavaScript strings are UTF-16 encoded. Characters outside the Basic Multilingual Plane (such as emoji) require two UTF-16 code units — called a **surrogate pair** — which means their `length` is 2.

**Key concepts:**

- **Code points** uniquely identify characters: `U+1F968` = 🥨
- **Escape sequences**: `\u{1F968}` (ES2015+ preferred) or legacy `🥨` (surrogate pair)
- `String.fromCodePoint(0x1F968)` → `'🥨'`
- `'🥨'.codePointAt(0).toString(16)` → `'1f968'`
- `'🥨'.length` → `2` (surrogate pair — the character takes two UTF-16 code units)

**Normalization** matters for assembled characters:

```javascript
const composed = 'Ñ';  // N + combining tilde
const single   = 'Ñ';        // precomposed single character

composed === single           // false — different byte sequences
composed.normalize() === single.normalize()  // true — same canonical form
```

**Advanced topics:**
- **Surrogate pairs** — characters above U+FFFF stored as two code units; `for...of` and spread handle them correctly, `.length` does not
- **Assembled characters** — a single visible character composed of multiple code points (e.g. family emoji `👨‍👩‍👧‍👦` uses zero-width joiners)

### Demo: Strings

See [`01-strings-demos/`](01-strings-demos/) for the full project. Each subdirectory is a standalone Node.js script runnable with `node index.js`.

**1 – running code** — minimal Node.js setup sanity check.

**2 – string templates** — complete template literal with multi-line, ternary, and arithmetic expressions:

```javascript
const message = `Hello ${firstName}!
Your cart contains ${cartItem.quantity} ticket${cartItem.quantity > 1 ? 's' : ''} for ${cartItem.event}
for a total of $${(cartItem.quantity * cartItem.price).toFixed(2)}.`;
```

**3 – tagged templates** — tag function that wraps interpolated values in `<em>` tags, demonstrating how `strings` and `values` interleave.

**4 – string methods** — `trim`, `slice`, `includes`, `startsWith` in action.

**5 – unicode** — `length` surprise on emoji, `fromCodePoint` / `codePointAt` round-trip, and `normalize()` equality check for assembled characters.

## Module 02: Control Flow

Modern JavaScript offers concise control flow patterns — short-circuit evaluation, nullish coalescing, optional chaining, and guard clauses — that reduce noise and defensive boilerplate.

### Truthy and Falsy Values

Every JavaScript value is either truthy or falsy in a boolean context. The complete list of **falsy** values:

- `0`
- `''` (empty string)
- `null`
- `undefined`
- `NaN`
- `false`

Everything else is truthy — including `0n` (BigInt zero is falsy), `[]`, `{}`, `'false'`, and `Infinity`.

```javascript
if (expression) {
    console.log('truthy');
} else {
    console.log('falsy');
}
```

### Short-Circuit Evaluation

Logical operators `&&` and `||` stop as soon as the result is determined — they return the actual value, not a boolean.

```javascript
// || returns the first truthy value (or the last value)
const selectedCategory = category || 'All';
// If category is '' (falsy), selectedCategory = 'All'
// If category is 'Rock' (truthy), selectedCategory = 'Rock'

// ?? (nullish coalescing) returns the first non-null/undefined value
const selectedQuantity = quantity ?? 1;
// If quantity is 0 (falsy but not null/undefined), selectedQuantity = 0
// If quantity is null/undefined, selectedQuantity = 1
```

**The key difference between `||` and `??`:**
- `||` treats all falsy values (`0`, `''`, `false`, `null`, `undefined`) as "missing"
- `??` only treats `null` and `undefined` as "missing" — `0` and `''` are valid values

### Optional Chaining and Nullish Coalescing

`?.` short-circuits to `undefined` instead of throwing when a property in the chain is `null` or `undefined`. Combine with `??` to supply a default.

```javascript
const user = { name: 'Kim', urls: { personal: undefined, company: 'http://example.com' } };

const personal  = user.urls?.personal  ?? 'no personal homepage';
// user.urls exists → user.urls.personal = undefined → ?? → 'no personal homepage'

const company   = user.urls?.company   ?? 'no company homepage';
// user.urls exists → user.urls.company = 'http://example.com' → keep value

const usesHTTPS = user.urls?.company?.startsWith('https://');
// Optional chaining on method calls too — returns undefined if company is nullish
```

Works for property access (`?.`), method calls (`?.()`), and bracket access (`?.[key]`).

### Guard Clauses

Guard clauses check preconditions at the top of a function and return early on failure. This keeps the happy path at the lowest indentation level and eliminates deeply nested `if/else` trees.

**Before (nested):**
```javascript
function updateAppSettings(user, settings) {
    if (user) {
        if (user.isAdmin) {
            if (user.userLevel >= 255) {
                // apply settings
            }
        }
    }
}
```

**After (guard clauses):**
```javascript
function updateAppSettings(user, settings) {
    if (!user || !user.isAdmin || !user.userLevel || user.userLevel < 255) {
        return;
    }
    // apply settings — guaranteed valid state from here
}
```

Guard clause benefits:
- **Return early / fail fast** — invalid states exit immediately
- **Flatten nested code** — happy path stays at top level
- **Handle edge cases** — each guard isolates one failure mode

### Ternary Operators

A ternary is a compact `if/else` that produces a value — useful inside template literals and JSX where statements cannot appear.

```javascript
// condition ? valueIfTruthy : valueIfFalsy
const message = `ticket${ quantity !== 1 ? 's' : '' }`;
```

Use ternaries for simple value selection. Avoid nesting ternaries — a nested ternary is harder to read than an `if/else`.

### Pattern Matching (Coming Soon)

A TC39 proposal (Stage 2 as of 2025) will add native pattern matching to JavaScript — a cleaner alternative to long `if/else` or `switch` chains for structural checks:

```javascript
// Proposed syntax (not yet available in engines)
const hasAccess = match (user) {
    { isAdmin: true, userLevel: >= 255 and < 1024 }: true,
    default: false
};
```

Until then, use guard clauses, optional chaining, and destructuring for the same effect.

### Demo: Control Flow

See [`02-control-flow-demos/`](02-control-flow-demos/) for the full project.

**1 – truthy falsy** — cycles through `42`, `0`, `''`, `' '`, `42/0`, `42/'a'` to observe truthy/falsy classification.

**2 – short-circuit** — contrasts `||` (skips `0` as falsy) vs `??` (treats `0` as a valid value):

```javascript
const category = "";
const quantity = 0;

const selectedCategory = category || 'All';    // 'All' — empty string is falsy
const selectedQuantity  = quantity ?? 1;        // 0 — not null/undefined
```

**3 – optional chaining** — navigates a user object with a missing `personal` URL and an `http://` company URL, demonstrating `?.` and `??` composing cleanly.

**4 – guard clauses** — `updateAppSettings` returns immediately if any precondition fails, keeping the implementation flat.

## Module 03: Error Handling

Modern JavaScript adds `Promise.try()`, the `using` declaration for automatic resource disposal, and `Promise.allSettled()` for concurrent operations where partial failure is acceptable.

### Synchronous vs. Asynchronous Exceptions

A function may throw synchronously (e.g. `JSON.parse`) or return a rejected Promise. Mixing both in one call chain forces callers to handle both `try/catch` and `.catch()`:

```javascript
// Problem: parseEvents throws synchronously — a .catch() won't catch it
Promise.resolve()
    .then(() => parseEvents(data))  // must be inside .then() to be caught
    .catch(error => console.error(error));
```

`Promise.try()` wraps any callback — sync or async — and normalises exceptions into a rejected Promise:

```javascript
// Before: try/catch required
try {
    const events = parseEvents(data);
} catch (error) {
    console.error(error);
}

// After: uniform Promise chain regardless of throw vs reject
Promise.try(() => parseEvents(data))
    .then(events => { /* ... */ })
    .catch(error => console.error(error));
```

### The `using` Declaration

The `using` declaration (TC39 Explicit Resource Management proposal, Stage 4) automatically calls `[Symbol.dispose]()` — or `[Symbol.asyncDispose]()` for `await using` — when the variable goes out of scope. This eliminates `try/finally` cleanup boilerplate.

```javascript
// Before: explicit try/finally to guarantee close()
const file = await open('log.txt', 'w');
try {
    await file.writeFile('Globoticket started' + EOL);
} finally {
    await file.close();
}

// After: file is closed automatically when the block exits
await using file = await open('log.txt', 'w');
await file.writeFile('Globoticket started' + EOL);
```

Properties of the `using` declaration:
- **Local variable** — scoped to the enclosing block
- **Scoped to block** — disposal fires at block exit, even on exception
- **Disposed once out of scope** — no explicit `close()`/`dispose()` needed

`using` works with any object that implements `[Symbol.dispose]`. `await using` works with `[Symbol.asyncDispose]`.

### Promise Concurrency: `Promise.all()` vs. `Promise.allSettled()`

| | `Promise.all()` | `Promise.allSettled()` |
|---|---|---|
| **Behaviour** | Rejects when any promise rejects | Waits for all to settle |
| **Resolution** | Array of values (if all fulfilled) | Array of `{status, value}` / `{status, reason}` objects |
| **Use when** | All operations must succeed | Interested in results of all, even partial failures |

```javascript
const uploads = await Promise.allSettled([
    uploadFile('band-photo.jpg'),
    uploadFile('band-logo.png'),
    uploadFile('terms-and-conditions.pdf')
]);

for (const upload of uploads) {
    if (upload.status === 'fulfilled') {
        console.log(upload.value);
    } else {
        console.error(upload.reason.message);
    }
}
```

**Other Promise combinators:**
- `Promise.any()` — resolves with the first fulfilled promise; rejects only if all reject
- `Promise.race()` — settles with the first promise to settle (fulfilled or rejected)

### Demo: Error Handling

See [`03-error-handling-demos/`](03-error-handling-demos/) for the full project. Organised as Before/After pairs showing the improvement.

**Before/1 – Promise.try** — `parseEvents` throws synchronously; wrapped in `try/catch` to handle it.

**After/1 – Promise.try** — same function call wrapped in `Promise.try()`, errors handled uniformly in `.catch()`.

**Before/2 – using** — `fs/promises open()` with explicit `try/finally` to close the file handle.

**After/2 – using** — `await using` replaces `try/finally`; the file closes automatically when the block exits.

**After/3 – Promise.allSettled** — three simulated file uploads with random failures; `allSettled` processes all results regardless of which uploads failed.

## Module 04: Iteration Patterns

Array higher-order methods (`map`, `filter`, `reduce`, `find`, `some`, `every`) and `for...of` replace manual index loops. Choosing the right method signals intent and enables short-circuit optimisation.

### Transforming Data: `map()`, `filter()`, `reduce()`

| Method | Input → Output | Use when |
|--------|---------------|----------|
| `map()` | array → same-length array | Transform every element |
| `filter()` | array → shorter (or equal) array | Remove elements that don't match |
| `reduce()` | array → single value | Aggregate: sum, object, string, etc. |

```javascript
const cart = [
    { quantity: 1, event: "The Network Squad", price: 33.99 },
    { quantity: 2, event: "Alexander Lemtov",  price: 39   },
    { quantity: 4, event: "Mariam Johnson",    price: 24.9 }
];

// map — add a computed 'total' property to each item
const totals = cart.map(item => ({ ...item, total: item.quantity * item.price }));

// filter — keep only items with quantity <= 2
const small = cart.filter(item => item.quantity <= 2);

// reduce — sum all item totals
const grandTotal = cart.reduce((sum, item) => sum + item.quantity * item.price, 0);
```

All three return a **new array or value** — they never mutate the original.

### Short-Circuiting Iteration: `find()`, `some()`, `every()`

These methods stop iterating as soon as the answer is known — important for performance on large arrays.

| Method | Returns | Stops when |
|--------|---------|-----------|
| `find()` | First matching element (or `undefined`) | First match found |
| `some()` | `true` if any element matches | First match found |
| `every()` | `true` if all elements match | First non-match found |

```javascript
const cart = [
    { event: "The Network Squad", price: 33.99, minAge: 6  },
    { event: "Alexander Lemtov",  price: 39,    minAge: 16 },
    { event: "Mariam Johnson",    price: 24.9,  minAge: 0  }
];

// some — is there at least one 18+ event?
const isAdultRequired = cart.some(item => item.minAge >= 18);    // false

// every — are ALL events under 18?
const allUnder18 = cart.every(item => item.minAge < 18);          // true

// Equivalent guard using !every
const adultRequired = !cart.every(item => item.minAge < 18);     // false
```

### `for...of` for Imperative Control

`for...of` iterates over any iterable (arrays, strings, Maps, Sets, generators) giving direct access to values. Use it when you need `break`, `continue`, or `await` inside the loop body — none of which work cleanly inside `map`/`filter` callbacks.

```javascript
// Value iteration (preferred over for...in for arrays)
for (const item of cart) {
    console.log(`${item.quantity} ticket(s) for ${item.event.toUpperCase()}, $${item.price} each.`);
}

// With index via entries()
for (const [index, item] of cart.entries()) {
    console.log(index, item.event);
}

// Works correctly on strings with multi-codepoint characters
for (const char of "👨‍👩‍👧‍👦") {
    console.log(char);  // emits the full family emoji, not surrogate halves
}
```

`for...in` iterates over **enumerable property keys** (strings) and includes inherited properties — avoid it for arrays.

**Additional iterator helpers** (available on iterator objects, Node 22+):
- `drop(n)` — skip first n items
- `take(n)` — take first n items  
- `flatMap(fn)` — map + flatten one level
- `toArray()` — materialise iterator to array

### Demo: Iteration Patterns

See [`04-iteration-patterns-demos/`](04-iteration-patterns-demos/) for the full project.

**1 – map** — adds a computed `total` field to each cart item using spread and `map`.

**2 – filter** — keeps only items with `quantity <= 2`.

**3 – reduce** — computes the grand total of the cart as a single number.

**4 – find** — (empty demo stub; `find` returns the first matching object reference).

**5 – some** — checks whether any item requires an adult (age ≥ 18).

**6 – every** — uses `!every(item => item.minAge < 18)` to express "not all items are under-18".

**7 – for...of** — iterates cart items and then a family emoji string, showing correct multi-codepoint character handling.

## Module 05: Data Collections

JavaScript's built-in `Map` and `Set` fill gaps that plain objects and arrays can't address cleanly: non-string keys, guaranteed insertion order, uniqueness enforcement, and O(1) membership tests.

### Why More Collection Types?

Plain objects (`{}`) have limitations:
- Keys must be strings or Symbols — objects as keys silently become `"[object Object]"`
- No guaranteed insertion order in older engines (modern engines maintain it for string keys)
- `in` and `hasOwnProperty` checks are verbose

Arrays (`[]`) lack:
- O(1) membership tests (`.includes()` is O(n))
- Automatic deduplication

`Map` and `Set` solve these problems directly.

### `Map` — Key-Value Collection

`Map` is a key-value collection where **any data type** can be a key — including objects, functions, and primitives. Insertion order is maintained.

```javascript
const event1 = { artist: 'The Firewall Squad', venue: 'Arena' };
const event2 = { artist: 'Staff Choir',        venue: 'Stadium' };

const ticketInventory = new Map();
ticketInventory.set(event1, 20);   // object as key
ticketInventory.set(event2, 1);

ticketInventory.has(event2);        // true
ticketInventory.size;               // 2
ticketInventory.delete(event2);
ticketInventory.size;               // 1

// Iterable with destructuring
for (const [event, quantity] of ticketInventory) {
    console.log(quantity, 'ticket(s) for', event);
}
```

**Map properties:**
- **Key-value collection** — each key maps to exactly one value
- **Any data type possible (even for the key)** — object identity used for equality
- **Order of elements is maintained** — guaranteed insertion order

### `Set` — Collection of Unique Values

`Set` stores unique values of any type. Adding a duplicate is a no-op. Insertion order is maintained.

```javascript
const waitingRoom = new Set();

waitingRoom.add(1);
waitingRoom.add(42);
waitingRoom.add(99);
waitingRoom.size;   // 3

waitingRoom.add(42);  // duplicate — ignored
waitingRoom.size;   // still 3

waitingRoom.delete(42);

for (const user of waitingRoom) {
    console.log(user);   // 1, 99
}
```

**Set properties:**
- **Collection of unique values** — duplicate entries are silently dropped
- **Any data type possible** (as long as there are no duplicates — uses same-value-zero equality)
- **Order of elements is maintained** — iteration follows insertion order

### Weak Collections: `WeakMap` and `WeakSet`

Weak collections hold object references *weakly* — if no other reference to an object exists, the garbage collector can reclaim it even if it is still in the collection. This makes them useful for metadata caches tied to object lifetime.

| | Strong (`Map`, `Set`) | Weak (`WeakMap`, `WeakSet`) |
|---|---|---|
| **Types** | `Map`, `Set` | `WeakMap`, `WeakSet` |
| **Key/value types** | Any data type | Objects only |
| **Iterable** | Yes | No |
| **Has size** | Yes | No |

```javascript
const userProfile = new WeakMap();
let user = { name: 'Tom' };

userProfile.set(user, { lastSeen: new Date(), language: 'en' });
userProfile.has(user);   // true

const waitingRoom = new WeakSet();
waitingRoom.add(user);
waitingRoom.has(user);   // true

// When user is nulled, both collections lose their reference
// The object becomes eligible for GC
user = null;
userProfile.has(user);   // false (user is now null)
```

Use `WeakMap` for per-object metadata (e.g. caching DOM node data) and `WeakSet` for tracking object membership without preventing GC.

### Demo: Data Collections

See [`05-data-collections-demos/`](05-data-collections-demos/) for the full project.

**1 – Map** — builds a ticket inventory keyed by object references (`event1`, `event2`), demonstrates `has`, `size`, `delete`, and `for...of` destructuring.

**2 – Set** — waiting room with `add` (including a duplicate that is ignored), `size`, `delete`, and iteration.

**3 – WeakMap WeakSet** — stores per-user profile metadata in a `WeakMap` and tracks presence in a `WeakSet`; sets `user = null` to demonstrate that the weak references are dropped (note: actual GC timing is non-deterministic).

## Module 06: Functional Programming Essentials

JavaScript supports multiple function syntaxes, closures for encapsulation, function factories for parameterised behaviour, and currying for partial application and function composition.

### Function Syntax

JavaScript has three function syntaxes that are equivalent in many contexts but differ in `this` binding and hoisting:

```javascript
// Function declaration — hoisted, has its own 'this'
function getTotal(item) {
    return item.quantity * item.price;
}

// Function expression — not hoisted, has its own 'this'
const getTotal = function(item) {
    return item.quantity * item.price;
};

// Arrow function — not hoisted, captures surrounding 'this'
const getTotal = item => item.quantity * item.price;
```

| | Function declaration / expression | Arrow function |
|---|---|---|
| **Hoisting** | Yes (declarations) / No (expressions) | No |
| **Return** | Explicit `return` required | Implicit for single expressions |
| **`this`** | Own `this` (dynamic) | Captures surrounding `this` (lexical) |
| **Use for** | Helper functions, APIs | Callbacks, array methods |

### Closures

A closure is a function that retains access to the variables of its enclosing scope even after that scope has finished executing. The inner function "closes over" the outer variable.

```javascript
function waitingRoom() {
    let waitTime = 0;       // outer variable

    return function() {
        return ++waitTime;  // inner function closes over waitTime
    };
}

const room = waitingRoom();
room();   // 1 — waitTime persists between calls
room();   // 2
```

Each call to `waitingRoom()` creates a **new independent closure** — a separate `waitTime` variable. `room` is a function but it carries its own private state.

### Private State via Closures

Returning an object from a closure exposes a controlled public API while keeping the underlying state inaccessible from outside — JavaScript's idiomatic module pattern before `class` private fields.

```javascript
function waitingRoom() {
    let waitTime = 0;       // private — unreachable from outside

    return {
        refresh() {
            waitTime++;
        },
        getState() {
            return waitTime;
        }
    };
}

const room = waitingRoom();
room.refresh();
room.refresh();
room.getState();    // 2

// room.waitTime is undefined — no direct access
```

### Function Factories

A function factory is a function that returns a new function configured by the factory's argument. Each returned function closes over its own copy of the configuration value.

```javascript
function waitingRoom(step) {
    let waitTime = 0;

    return function() {
        waitTime += step;   // step is closed over — fixed at factory call time
        return waitTime;
    };
}

const room = waitingRoom(7);   // step = 7
room();   // 7
room();   // 14
room();   // 21

const fastRoom = waitingRoom(2);  // independent — its own step and waitTime
fastRoom();   // 2
```

The key insight: `step` is captured by value at the time `waitingRoom(step)` is called. Each factory call produces an independent function with its own closed-over state.

### Currying

Currying transforms a function with multiple arguments into a chain of single-argument functions. Each call returns a new function until all arguments have been supplied.

```javascript
// Non-curried
const multiply = (a, b) => a * b;
multiply(3, 4);   // 12

// Curried
const curriedMultiply = a => b => a * b;
const triple = curriedMultiply(3);   // partially applied — 'a' is fixed as 3
triple(4);   // 12
triple(7);   // 21

// Applied inline
curriedMultiply(3)(4);   // 12
```

Benefits of currying:
- **Multiple arguments → multiple functions** — each function takes one argument
- **Allows code reuse** — partially apply to create specialised functions (`triple`, `double`)
- **Allows composition** — curried functions can be piped together cleanly

Currying is the foundation of functional composition pipelines in libraries like Ramda and fp-ts.

### Demo: Functional Programming Essentials

See [`06-functional-programming-essentials-demos/`](06-functional-programming-essentials-demos/) for the full project.

**1 – function syntax** — three implementations of `getTotal` (declaration, expression, arrow) all produce the same result, showing the syntactic options.

**2 – closures** — `waitingRoom()` returns an inner function; calling `room()` repeatedly increments the closed-over `waitTime`.

**3 – private state** — same `waitingRoom` pattern but returning an object with `refresh()` and `getState()` to control access to the internal counter.

**4 – function factory** — `waitingRoom(step)` configures the increment size; the demo uses `step = 7` to produce 7, 14, 21, … showing each call uses the factory-configured step.

**5 – currying** — two sub-examples showing real-world currying:

```javascript
// filter/index.js — curried predicate for filtering by ID
const idFilter = (id) => (user) => user.id === id;
const filterById = (elements, id) => elements.filter(idFilter(id));

// log/index.js — generic curry() wrapper, then partial application
function curry(fn) {
    return function(one) {
        return function(two) { return fn(one, two); };
    };
}
const curryLog = curry(log);
const criticalLog = curryLog('critical');   // level fixed, message still free
criticalLog('user does not exist');
```